# Kaggle Queue Runner
A stable bootstrap. Update workloads.json and scripts in the controlled public repository while this notebook waits. Checks and useful work share this allocation. Kaggle Secrets supply optional private networking and artifact credentials.

In [ ]:
from pathlib import Path
import subprocess,sys,uuid,os

REPOSITORY = "https://github.com/enigmatic-figure/kaggle-queue-runner.git"
BRANCH = "main"

# This clone happens AFTER Kaggle allocates the worker, so it sees queue-time edits.
runtime = Path('/tmp') / ('queue-bootstrap-' + uuid.uuid4().hex)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPOSITORY,str(runtime)],check=True)
commit = subprocess.check_output(['git','-C',str(runtime),'rev-parse','HEAD'],text=True).strip()
print({'bootstrap_commit':commit},flush=True)
# Isolate runner/storage dependencies from the Kaggle model/JAX environment.
engine = runtime / '.engine'
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(engine)],check=True)
python = engine / 'bin' / 'python'
subprocess.run([str(python),'-m','pip','install','--quiet','-e',str(runtime)+'[hub]'],check=True)
environment = {**os.environ, 'QUEUE_WORKLOAD_PYTHON': sys.executable}
subprocess.run([str(python),'-u','-m','queue_runner.cli','worker',
    '--repo',REPOSITORY,'--branch',BRANCH,'--hours','8','--idle-minutes','5',
    '--network','auto'],cwd=runtime,env=environment,check=True)
